# Этап 4: Чанкование

**Стратегия:**
1. Логическая нарезка по Markdown-заголовкам (`MarkdownHeaderTextSplitter`)
2. Рекурсивная нарезка длинных секций (`RecursiveCharacterTextSplitter`)
3. Фильтрация слишком коротких чанков (< 10 токенов)

**Принцип разделения текста и метаданных:**
- `text` -- чистый контент чанка (только для эмбеддинга)
- `metadata` -- структурированный dict с заголовками секций + метаданные статьи из CSV
- Метаданные НЕ вклеиваются в текст, а сохраняются отдельно для Qdrant payload

In [1]:
from pathlib import Path
from datasets import Dataset
import pandas as pd
import os
import logging

from transformers import AutoTokenizer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [2]:
# --- КОНФИГ ---
MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"
INPUT_FILE = Path('../../data/processed/parquet/processed_data.parquet')
METADATA_CSV = Path('../../data/metadata/arxiv_NLP_2021_2026_metadata.csv')
OUTPUT_PARQUET = Path('../../data/processed/all_chunks.parquet')

CHUNK_SIZE_TOKENS = 512
CHUNK_OVERLAP_TOKENS = 50
MIN_CHUNK_TOKENS = 10  # Минимальная длина чанка в токенах

BATCH_SIZE = 100

## 1. Загрузка данных и метаданных

In [3]:
# --- Загрузка основного датасета ---
ds = Dataset.from_parquet(str(INPUT_FILE))
logger.info(f"Dataset loaded: {len(ds)} documents")

2026-06-05 14:33:46,356 - INFO - Dataset loaded: 825 documents


In [4]:
# --- Загрузка метаданных статей из CSV ---
# Подтягиваем title, authors, published для обогащения payload в Qdrant
meta_df = pd.read_csv(
    METADATA_CSV,
    usecols=['arxiv_id', 'title', 'authors', 'published'],
    dtype={'arxiv_id': str, 'title': str, 'authors': str, 'published': str}
)

# Строим lookup: doc_id -> {title, authors, published}
paper_metadata = (
    meta_df
    .dropna(subset=['arxiv_id'])
    .set_index('arxiv_id')
    [['title', 'authors', 'published']]
    .to_dict(orient='index')
)

logger.info(f"Paper metadata loaded: {len(paper_metadata)} entries")

# Проверка пересечения
doc_ids_in_ds = set(ds['doc_id'])
doc_ids_in_meta = set(paper_metadata.keys())
coverage = len(doc_ids_in_ds & doc_ids_in_meta) / len(doc_ids_in_ds) * 100
logger.info(f"Metadata coverage: {coverage:.1f}% of documents have metadata")

2026-06-05 14:33:50,306 - INFO - Paper metadata loaded: 12139 entries
2026-06-05 14:33:50,316 - INFO - Metadata coverage: 100.0% of documents have metadata


## 2. Инициализация сплиттеров

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "header_1"),
        ("##", "header_2"),
        ("###", "header_3")
    ]
)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=CHUNK_SIZE_TOKENS,
    chunk_overlap=CHUNK_OVERLAP_TOKENS
)

logger.info(f"Splitters ready: chunk_size={CHUNK_SIZE_TOKENS} tokens, overlap={CHUNK_OVERLAP_TOKENS} tokens")

2026-06-05 14:35:13,979 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-05 14:35:14,491 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/config.json "HTTP/1.1 200 OK"
2026-06-05 14:35:14,901 - INFO - HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-05 14:35:15,420 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-Embedding-0.6B/97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-05 14:35:15,828 - INFO - HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen3-Embedding-0.6B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-06-05 14:35:16,232 - INFO - HTTP Request: GET https://huggin

## 3. Функция чанкования

**Что собираем:**
- `text` -- ТОЛЬКО чистый контент (`chunk.page_content`). Идёт в Qdrant payload и подаётся в LLM.
- `embed_text` -- путь заголовков секции + текст (`header_1 > header_2 > header_3\n\ntext`). Контекстуализированный вариант для эмбеддинга: чанк из середины статьи получает топик-якорь. Если заголовков нет -- равен `text`.
- `metadata` -- заголовки секций по отдельности + метаданные статьи (title, authors, published) из CSV. Уедет в payload, а НЕ в эмбеддинг.

**Зачем два текстовых поля:** позволяет честно сравнить retrieval «с заголовками vs без» на golden-сете (нотбук 10), не перегоняя данные заново. В нотбуке 05 эмбедим `embed_text`; чистый `text` остаётся для показа и для LLM.

In [6]:
def process_batch(batch, md_splitter, text_splitter, tokenizer, paper_metadata):
    """Чанкование батча документов с разделением текста и метаданных.

    Текст чанка -- чистый контент без заголовков.
    Метаданные (заголовки секций + info о статье) хранятся отдельно.

    Дополнительно собираем `embed_text` -- путь заголовков секции + текст.
    Это контекстуализированный вариант для эмбеддинга: чанк из середины
    статьи получает топик-якорь (header_1/2/3). Исходный `text` остаётся
    чистым -- он идёт в payload и подаётся в LLM.

    Все зависимости определены внутри функции,
    т.к. num_proc > 1 порождает отдельные процессы без доступа к глобалам Jupyter.

    Returns:
        dict с ключами: text, embed_text, doc_id, metadata
    """
    import logging
    logger = logging.getLogger(__name__)

    MIN_CHUNK_TOKENS = 10

    chunk_texts = []
    chunk_embed_texts = []
    chunk_doc_ids = []
    chunk_metadata_list = []

    for doc_id, md_content in zip(batch['doc_id'], batch['md']):
        # Пропускаем невалидные документы
        if not md_content or not isinstance(md_content, str) or len(md_content.strip()) == 0:
            continue

        try:
            # 1. Нарезка по Markdown-заголовкам
            sections = md_splitter.split_text(md_content)

            # 2. Рекурсивная нарезка длинных секций
            chunks = text_splitter.split_documents(sections)

            # 3. Метаданные статьи из CSV (title, authors, published)
            paper_meta = paper_metadata.get(doc_id, {})

            for chunk_idx, chunk in enumerate(chunks):
                text = chunk.page_content.strip()

                # Фильтрация слишком коротких чанков (по токенам)
                if len(tokenizer.encode(text, add_special_tokens=False)) < MIN_CHUNK_TOKENS:
                    continue

                # Метаданные: заголовки секций из MarkdownHeaderTextSplitter
                header_1 = chunk.metadata.get('header_1', '')
                header_2 = chunk.metadata.get('header_2', '')
                header_3 = chunk.metadata.get('header_3', '')

                # Контекстуализированный текст для эмбеддинга:
                # путь непустых заголовков секции + сам текст чанка
                header_path = " > ".join(h for h in (header_1, header_2, header_3) if h)
                embed_text = f"{header_path}\n\n{text}" if header_path else text

                # Собираем metadata dict
                meta = {
                    # Заголовки секций (структура документа)
                    'header_1': header_1,
                    'header_2': header_2,
                    'header_3': header_3,
                    # Метаданные статьи из arXiv API
                    'title': paper_meta.get('title', ''),
                    'authors': paper_meta.get('authors', ''),
                    'published': paper_meta.get('published', ''),
                    # Позиция чанка в документе
                    'chunk_index': chunk_idx,
                }

                chunk_texts.append(text)
                chunk_embed_texts.append(embed_text)
                chunk_doc_ids.append(doc_id)
                chunk_metadata_list.append(meta)

        except Exception as e:
            logger.error(f"Error processing doc_id {doc_id}: {e}")
            continue

    return {
        'text': chunk_texts,
        'embed_text': chunk_embed_texts,
        'doc_id': chunk_doc_ids,
        'metadata': chunk_metadata_list,
    }

## 4. Запуск чанкования

In [7]:
chunked_ds = ds.map(
    process_batch,
    batched=True,
    batch_size=BATCH_SIZE,
    remove_columns=ds.column_names,
    num_proc=os.cpu_count(),
    fn_kwargs={
        'md_splitter': md_splitter,
        'text_splitter': text_splitter,
        'tokenizer': tokenizer,
        'paper_metadata': paper_metadata,
    }
)

logger.info(f"Chunking complete: {len(chunked_ds)} chunks")

Map (num_proc=12):   0%|          | 0/825 [00:00<?, ? examples/s]

2026-06-05 14:36:12,904 - INFO - Chunking complete: 33307 chunks


## 5. Проверка результатов

In [8]:
# Выборочная проверка: text -- чистый, embed_text -- с заголовками, метаданные -- отдельно
sample = chunked_ds[0]
print('=== SAMPLE CHUNK ===')
print(f"doc_id:     {sample['doc_id']}")
print(f"text:       {sample['text'][:200]}...")
print(f"embed_text: {sample['embed_text'][:200]}...")
print(f"metadata:   {sample['metadata']}")
print()

# Статистика длин чанков в токенах (батчами, чтобы не держать всё в RAM)
STATS_BATCH = 1000
token_lengths = []
for i in range(0, len(chunked_ds), STATS_BATCH):
    texts = chunked_ds[i:i + STATS_BATCH]['text']
    encoded = tokenizer(texts, add_special_tokens=False)['input_ids']
    token_lengths.extend(len(ids) for ids in encoded)

print(f"Total chunks:      {len(chunked_ds)}")
print(f"Avg chunk length:  {sum(token_lengths) / len(token_lengths):.0f} tokens")
print(f"Min chunk length:  {min(token_lengths)} tokens")
print(f"Max chunk length:  {max(token_lengths)} tokens")

# Сколько чанков реально обогатилось заголовками (embed_text != text)
enriched = sum(1 for t, e in zip(chunked_ds['text'], chunked_ds['embed_text']) if e != t)
print(f"Chunks with headers in embed_text: {enriched}/{len(chunked_ds)} ({enriched/len(chunked_ds)*100:.1f}%)")

# Проверка, что title подтянулся
titles_filled = sum(1 for m in chunked_ds['metadata'] if m.get('title'))
print(f"Chunks with title: {titles_filled}/{len(chunked_ds)} ({titles_filled/len(chunked_ds)*100:.1f}%)")

=== SAMPLE CHUNK ===
doc_id:     2512.23988v1
text:       ###### Abstract  
Despite the growing reasoning capabilities of recent large language models (LLMs), their internal mechanisms during the reasoning process remain underexplored. Prior approaches often...
embed_text: Fantastic Reasoning Behaviors and Where to Find Them: Unsupervised Discovery of the Reasoning Process

###### Abstract  
Despite the growing reasoning capabilities of recent large language models (LLM...
metadata:   {'authors': 'Zhenyu Zhang, Shujian Zhang, John Lambert, Wenxuan Zhou, Zhangyang Wang, Mingqing Chen, Andrew Hard, Rajiv Mathews, Lun Wang', 'chunk_index': 0, 'header_1': 'Fantastic Reasoning Behaviors and Where to Find Them: Unsupervised Discovery of the Reasoning Process', 'header_2': '', 'header_3': '', 'published': '2025-12-30', 'title': 'Fantastic Reasoning Behaviors and Where to Find Them: Unsupervised Discovery of the Reasoning Process'}

Total chunks:      33307
Avg chunk length:  343 tokens
Min ch

## 6. Сохранение

In [9]:
OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
chunked_ds.to_parquet(str(OUTPUT_PARQUET))

logger.info(f"Saved to {OUTPUT_PARQUET}: {len(chunked_ds)} chunks")

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

2026-06-05 14:36:41,812 - INFO - Saved to ../../data/processed/all_chunks.parquet: 33307 chunks
